    Once we have constructed the full BdG hamiltonian we can diagonalize it. However, we need sorted eigenvectors and eigenvalues. We will introduce those here.

In [1]:
using LinearAlgebra
using PyCall
using CSV
using DataFrames
using Plots
using DelimitedFiles
@pyimport numpy as np

In [2]:
# Importing personal modules
include("src/bdg_utilities.jl")
include("src/general_utils.jl")

Main.GeneralUtils

In [3]:
# Loading personal modules
using .BdGUtilities: generate_t_map, extract_geometry, build_H_twisted, generate_t_matrix
using .GeneralUtils: fermi_fn, show_all, show_matrix

using .BdGUtilities: create_bdg_ham_full, create_bdg_ham_full_twisted, create_bdg_ham_up, create_bdg_ham_dn

f = fermi_fn

fermi_fn (generic function with 2 methods)

In [4]:
"""
    sort_evecs(evector, nSites)

From a 2n x 2n BdG eigenvector matrix, whose columns are eigenvectors,
extract the positive-energy u and v blocks in shape (nSites x nSites).

Returns:
- `un`, `vn`: matrices of size (nSites x nSites), where `un[n,i]` and `vn[n,i]`
  correspond to the positive-energy state `n`
"""
function sort_evecs(evector::AbstractMatrix{<:Number}, nSites::Int)
    # numpy eigh returns eigenvectors in columns: evecs[:,k] is k-th eigenvector.
    # We want "state index" as first dimension: (nStates × 2nSites).
    evecsT = transpose(evector)

    # For a standard BdG ordering, positive energies are the last nSites states
    epos = evecsT[nSites+1:2*nSites, :]  # (nSites × 2nSites)

    un = epos[:, 1:nSites]
    vn = epos[:, nSites+1:2*nSites]
    return un, vn
end



"""
    sort_evecs_spinfull(evecs, evals, nSites; tol=0.0)

Select the positive-energy BdG eigenvectors and separate them into
(u_up, v_dn, u_dn, v_up) for the basis (c_up; c_dn^dagger; c_dn; c_up^dagger).

The output is arranged so that `u_up[n,i]` and the other components can be used
directly in `calc_delta_spinfull`.
"""
function sort_evecs_spinfull(
    evecs::AbstractMatrix{<:Number},
    evals::AbstractVector{<:Real},
    nSites::Int; tol::Real=0.0
)
    @assert size(evecs,1) == 4nSites && size(evecs,2) == 4nSites
    @assert length(evals) == 4nSites

    idx = findall(>(tol), evals)          # positive-energy indices
    VposT = transpose(evecs[:, idx])      # (nPos)×(4n) with no conjugation

    u_up = VposT[:, 1:nSites]
    v_dn = VposT[:, nSites+1 : 2nSites]
    u_dn = VposT[:, 2nSites+1 : 3nSites]
    v_up = VposT[:, 3nSites+1 : 4nSites]

    return u_up, v_dn, u_dn, v_up, evals[idx]
end



"""
    check_norm(un, vn, nSites; digits=5)

Check the normalization |un|^2 + |vn|^2 = 1 for each eigenstate.

Arguments:
- `un`, `vn`: particle and hole components
- `nSites`: number of eigenstates
- `digits`: rounding precision

Returns:
- vector of normalization values
"""
function check_norm(
    un::AbstractMatrix{<:Number},
    vn::AbstractMatrix{<:Number},
    nSites::Int; digits::Int=5
)
    normList = zeros(Float64, nSites)
    for n in 1:nSites
        normList[n] =
            sum(abs2, un[n, :]) +
            sum(abs2, vn[n, :])
    end
    return round.(normList, digits=digits)
end



"""
    check_norm_spinful(u_up, v_dn, u_dn, v_up, nSites; digits=5)

Check the normalization sum(|u|^2 + |v|^2) over spin components
for each eigenstate.

Arguments:
- `u_up`, `v_dn`, `u_dn`, `v_up`: BdG components
- `nSites`: number of eigenstates
- `digits`: rounding precision

Returns:
- vector of normalization values
"""
function check_norm_spinful(
    u_up::AbstractMatrix{<:Number},
    v_dn::AbstractMatrix{<:Number},
    u_dn::AbstractMatrix{<:Number},
    v_up::AbstractMatrix{<:Number},
    nSites::Int; digits::Int=5
)
    normList = zeros(Float64, nSites)
    for n in 1:nSites
        normList[n] =
            sum(abs2, u_up[n, :]) +
            sum(abs2, v_dn[n, :]) +
            sum(abs2, u_dn[n, :]) +
            sum(abs2, v_up[n, :])
    end
    return round.(normList, digits=digits)
end

check_norm_spinful

In [5]:
N             = 2
nSites        = 4 * N ^ 2

nUp          = ones(Float64, nSites)
nDn          = ones(Float64, nSites)

df = CSV.read("data/N$(N)/raw_df_square-octagon$N.csv", DataFrame)

x, y, neighbors = extract_geometry(df)
lambda = 0.9
lambda_iso = 0.0
t2 = 1.0

tMap = generate_t_map(neighbors, t2=t2)
twist = 0.0

iso_map = nothing

if lambda == 0.0 && twist == 0.0
    deltaOld      = 0.7 * ones(Float64, nSites)
    isComplexCalc = false
else
    deltaOld      = 0.7 * ones(ComplexF64, nSites)
    isComplexCalc = true
end

impuritySite  = 3
mu             = -0.2
U             = 3

J             = 0.0
K             = 0.0
T             = 0.0

tMatFileName = "data/N$(N)/ham_$(N)_t2_$(t2)"
tMat = generate_t_matrix(tMatFileName); ## One doesn't need to use prebuilt hamiltonian. dataframe is enough to construct this matrix

In [6]:
tMap

Dict{Tuple{Int64, Int64}, Float64} with 64 entries:
  (16, 13) => 1.0
  (16, 14) => 1.0
  (15, 5)  => 1.0
  (11, 10) => 1.0
  (4, 6)   => 1.0
  (9, 3)   => 1.0
  (7, 8)   => 1.0
  (14, 12) => 1.0
  (14, 15) => 1.0
  (13, 7)  => 1.0
  (2, 1)   => 1.0
  (6, 5)   => 1.0
  (2, 8)   => 1.0
  (3, 9)   => 1.0
  (16, 10) => 1.0
  (5, 15)  => 1.0
  (7, 6)   => 1.0
  (1, 3)   => 1.0
  (9, 10)  => 1.0
  (1, 4)   => 1.0
  (14, 13) => 1.0
  (3, 4)   => 1.0
  (12, 11) => 1.0
  (14, 16) => 1.0
  (13, 15) => 1.0
  ⋮        => ⋮

In [7]:
H1 = create_bdg_ham_up(deltaOld, copy(tMat), mu, nSites, nUp, nDn, U, impuritySite, J;
                       isComplex=isComplexCalc, includeHartree=false)

H2 = create_bdg_ham_dn(deltaOld, copy(tMat), mu, nSites, nUp, nDn, U, impuritySite, J;
                       isComplex=isComplexCalc, includeHartree=false);

In [8]:
(evals1, evecs1) = np.linalg.eigh(H1)
un_up, vn_dn = sort_evecs(evecs1, nSites);
check_norm(un_up, vn_dn, nSites)

16-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0

In [9]:
(evals2, evecs2) = np.linalg.eigh(H2)
un_dn, vn_up = sort_evecs(evecs2, nSites);
check_norm(un_dn, vn_up, nSites)

16-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0

In [10]:
Hfull = create_bdg_ham_full(deltaOld, tMat, mu, nSites, nUp, nDn, U, impuritySite, J,
    x, y, neighbors, lambda, tMap)

64×64 Matrix{ComplexF64}:
 0.2+0.0im  1.0+0.0im  1.0+0.0im  …   0.0+0.0im   0.0+0.0im   0.0+0.0im
 1.0+0.0im  0.2+0.0im  1.0+0.0im      0.0+0.0im   0.0+0.0im   0.0+0.0im
 1.0+0.0im  1.0+0.0im  0.2+0.0im      0.0+0.0im   0.0+0.0im   0.0+0.0im
 1.0+0.0im  1.0+0.0im  1.0+0.0im      0.0+0.0im   0.0+0.0im   0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im      0.0+0.0im   0.0+0.0im   0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  …   0.0+0.0im   0.0+0.0im   0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im      0.0+0.0im   0.0+0.0im   0.0+0.0im
 0.0+0.0im  1.0+0.0im  0.0+0.0im      0.0+0.0im   0.0+0.0im   0.0+0.0im
 0.0+0.0im  0.0+0.0im  1.0+0.0im      0.0+0.0im   0.0+0.0im   0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im      0.0+0.0im   0.0+0.0im   0.0+0.0im
 1.0+0.0im  0.0+0.0im  0.0+0.0im  …   0.0+0.0im   0.0+0.0im   0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im      0.0+0.0im   0.0+0.0im   0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im      0.0+0.0im   0.0+0.0im   0.0+0.0im
    ⋮                             ⋱   